In [28]:
from pathlib import Path
import numpy as np

PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

VIDEO_FEATURE_PATH = (
    PROJECT_PATH
    / "datasets"
    / "processed"
    / "video_features"
)

SENSOR_DATASET_PATH = (
    PROJECT_PATH
    / "datasets"
    / "processed"
    / "uah_dataset.npz"
)

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
sensor_data = np.load(SENSOR_DATASET_PATH)

y = sensor_data["y"]
groups = sensor_data["groups"]

video_feature_files = sorted(
    VIDEO_FEATURE_PATH.glob("*.npy")
)

print(len(video_feature_files))

40


In [4]:
video_samples = []
labels = []
aligned_groups = []

for trip_id in range(40):

    video_trip = np.load(video_feature_files[trip_id])

    label_trip = y[groups == trip_id]

    common = min(
        len(video_trip),
        len(label_trip)
    )

    video_samples.append(
        video_trip[:common]
    )

    labels.append(
        label_trip[:common]
    )

    aligned_groups.append(
        np.full(common, trip_id)
    )

video_samples = np.concatenate(video_samples)
labels = np.concatenate(labels)
aligned_groups = np.concatenate(aligned_groups)

In [5]:
print(video_samples.shape)
print(labels.shape)

(30568, 2048)
(30568,)


In [6]:
from sklearn.model_selection import train_test_split

unique_groups = np.unique(aligned_groups)

train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=0.20,
    random_state=42,
)

train_mask = np.isin(
    aligned_groups,
    train_groups,
)

test_mask = np.isin(
    aligned_groups,
    test_groups,
)

X_train = video_samples[train_mask]
X_test = video_samples[test_mask]

y_train = labels[train_mask]
y_test = labels[test_mask]

In [8]:
import torch

In [9]:
from torch.utils.data import Dataset

class VideoDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

In [10]:
train_dataset = VideoDataset(
    X_train,
    y_train,
)

test_dataset = VideoDataset(
    X_test,
    y_test,
)

print(len(train_dataset))
print(len(test_dataset))

24103
6465


In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
)

print(len(train_loader))
print(len(test_loader))

754
203


In [12]:
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)

torch.Size([32, 2048])
torch.Size([32])


In [15]:
import sys

sys.path.append("/content/drive/MyDrive/UAH_Project/src")

In [16]:
import importlib

import video_model

importlib.reload(video_model)

from video_model import VideoClassifier

In [17]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = VideoClassifier().to(device)

print(model)

VideoClassifier(
  (network): Sequential(
    (0): Linear(in_features=2048, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=3, bias=True)
  )
)


In [34]:
import os

print(os.listdir("/content/drive/MyDrive/UAH_Project/src"))

['__init__.py', 'utils.py', '__pycache__', '_sync_test.txt', 'config.py', 'preprocessor.py', 'data_loader.py', 'trainer.py', 'lstm_model.py', 'fusion_model.py', 'fusion_trainer.py', 'video_model.py', 'video_trainer.py']


In [40]:
from pathlib import Path

print(Path("/content/drive/MyDrive/UAH_Project/src/video_trainer.py").exists())

True


In [68]:
import importlib.util

spec = importlib.util.find_spec("video_trainer")
print(spec)

None


In [71]:
import sys

print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/drive/MyDrive/UAH_Project/src', '/content/drive/MyDrive/UAH_Project/src']


In [72]:
import importlib.util

module_path = "/content/drive/MyDrive/UAH_Project/src/video_trainer.py"

spec = importlib.util.spec_from_file_location(
    "video_trainer",
    module_path,
)

video_trainer = importlib.util.module_from_spec(spec)

spec.loader.exec_module(video_trainer)

fit = video_trainer.fit

print("Video trainer imported successfully!")

Video trainer imported successfully!


In [74]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)

weights = torch.FloatTensor(
    class_weights
).to(device)

In [75]:
criterion = torch.nn.CrossEntropyLoss(
    weight=weights
)

In [76]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
)

In [77]:
history = fit(
    model,
    train_loader,
    test_loader,
    criterion,
    optimizer,
    device,
    epochs=20,
    patience=3,
)

Epoch 1/20 | Train Loss 0.7447 | Train Acc 0.6467 | Val Loss 1.9233 | Val Acc 0.2031
Epoch 2/20 | Train Loss 0.3659 | Train Acc 0.8467 | Val Loss 2.4155 | Val Acc 0.2049
Epoch 3/20 | Train Loss 0.2227 | Train Acc 0.9128 | Val Loss 3.1352 | Val Acc 0.1507
Epoch 4/20 | Train Loss 0.1436 | Train Acc 0.9476 | Val Loss 3.5981 | Val Acc 0.1876
Epoch 5/20 | Train Loss 0.0999 | Train Acc 0.9656 | Val Loss 3.7786 | Val Acc 0.1876

Early stopping!
Best Val Acc: 0.2049


In [78]:
# Her trip_id için hangi sürüş olduğunu göster
for trip_id in range(5):
    # Sensör tarafında bu trip'in hangi sürüş olduğu
    sensor_indices = np.where(groups == trip_id)[0]
    print(f"Trip {trip_id} sensör: {sensor_indices[0]} - {sensor_indices[-1]}")

    # Video tarafında ne yükleniyor
    print(f"Trip {trip_id} video: {video_feature_files[trip_id].name}")
    print()

Trip 0 sensör: 0 - 605
Trip 0 video: 20151110175712-16km-D1-NORMAL1-SECONDARY.npy

Trip 1 sensör: 606 - 1246
Trip 1 video: 20151110180846-16km-D1-NORMAL2-SECONDARY.npy

Trip 2 sensör: 1247 - 2101
Trip 2 video: 20151111123123-25km-D1-NORMAL-MOTORWAY.npy

Trip 3 sensör: 2102 - 2818
Trip 3 video: 20151111125204-24km-D1-AGGRESSIVE-MOTORWAY.npy

Trip 4 sensör: 2819 - 3748
Trip 4 video: 20151111132343-25km-D1-DROWSY-MOTORWAY.npy



In [79]:
# Test triplerinin hangi sürücülere ait olduğuna bak
test_trips = [19, 16, 15, 26, 4, 12, 37, 27]
for t in test_trips:
    print(video_feature_files[t].name)

20151126132012-16km-D3-DROWSY-SECONDARY.npy
20151126124207-16km-D3-NORMAL1-SECONDARY.npy
20151126113753-26km-D3-DROWSY-MOTORWAY.npy
20151204154908-25km-D4-AGGRESSIVE-MOTORWAY.npy
20151111132343-25km-D1-DROWSY-MOTORWAY.npy
20151120163335-16km-D2-AGGRESSIVE-SECONDARY.npy
20151221112444-D6-NORMAL-SECONDARY.npy
20151204160822-25km-D4-DROWSY-MOTORWAY.npy


In [81]:
# Train triplerinde hangi sürücüler var
train_trips = [t for t in range(40) if t not in test_trips]
for t in train_trips:
    name = video_feature_files[t].name
    driver = name.split('-')[2]  # D1, D2 gibi
    print(driver, name[:30])

D1 20151110175712-16km-D1-NORMAL1
D1 20151110180846-16km-D1-NORMAL2
D1 20151111123123-25km-D1-NORMAL-
D1 20151111125204-24km-D1-AGGRESS
D1 20151111134542-16km-D1-AGGRESS
D1 20151111135605-13km-D1-DROWSY-
D2 20151120131704-26km-D2-NORMAL-
D2 20151120133457-26km-D2-AGGRESS
D2 20151120135150-25km-D2-DROWSY-
D2 20151120160903-16km-D2-NORMAL1
D2 20151120162104-17km-D2-NORMAL2
D2 20151120165604-16km-D2-DROWSY-
D3 20151126110501-26km-D3-NORMAL-
D3 20151126125500-16km-D3-NORMAL2
D3 20151126130708-16km-D3-AGGRESS
D3 20151126134731-26km-D3-AGGRESS
D4 20151203171759-16km-D4-NORMAL1
D4 20151203173100-17km-D4-NORMAL2
D4 20151203174323-16km-D4-AGGRESS
D4 20151203175659-17km-D4-DROWSY-
D4 20151204152839-25km-D4-NORMAL-
D5 20151209151237-25km-D5-NORMAL-
D5 20151209153136-25km-D5-AGGRESS
D5 20151211160230-25km-D5-DROWSY-
D5 20151211162829-16km-D5-NORMAL1
D5 20151211164123-17km-D5-NORMAL2
D5 20151211165349-12km-D5-AGGRESS
D5 20151211170500-16km-D5-DROWSY-
D6 20151217162706-26km-D6-NORMAL-
D6 20151217164

In [82]:
from collections import Counter

label_names = {
    0: "NORMAL",
    1: "DROWSY",
    2: "AGGRESSIVE",
}

print("TRAIN")
train_counter = Counter(y_train)

for k, v in sorted(train_counter.items()):
    print(f"{label_names[k]:12s}: {v}")

print("\nTEST")
test_counter = Counter(y_test)

for k, v in sorted(test_counter.items()):
    print(f"{label_names[k]:12s}: {v}")

TRAIN
NORMAL      : 11536
DROWSY      : 6242
AGGRESSIVE  : 6325

TEST
NORMAL      : 1455
DROWSY      : 3496
AGGRESSIVE  : 1514


In [83]:
import pandas as pd

train_counter = Counter(y_train)
test_counter = Counter(y_test)

df = pd.DataFrame({
    "Train": [train_counter[i] for i in range(3)],
    "Test": [test_counter[i] for i in range(3)],
}, index=["NORMAL", "DROWSY", "AGGRESSIVE"])

df["Train %"] = (
    df["Train"] / df["Train"].sum() * 100
).round(2)

df["Test %"] = (
    df["Test"] / df["Test"].sum() * 100
).round(2)

print(df)

            Train  Test  Train %  Test %
NORMAL      11536  1455    47.86   22.51
DROWSY       6242  3496    25.90   54.08
AGGRESSIVE   6325  1514    26.24   23.42
